In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid

**DATA**

In [2]:
''' GEOMETRY & MASS '''
m_curb = 1195 #kg
m_driver = 75 #kg
m = m_curb + m_driver #kg
length = 3.999 #m
width = 1.775 #m
height = 1.597 #m
L = 2.570 #m
b_cg = 0.53*L #m
c_cg = 0.47*L #m
h_cg = 0.49 #m

'''TRANSMISSION & TIRE '''
# 19 inches rim; /39.37 converts inches in meters; 0.7*155 is thickness of tire
D_wheel = (( 19 / 39.37 ) + ( 155 * 0.7 * 2)*10**-3)
eta_t = 0.95 #-
i_gear = 9.665 #-
f0 = 0.008
f2 = 0.4e-7
R_e = 0.339 #m

''' ENGINE '''
P_1p = 125e3 #W
P_1n = 75e3 #W
C_max = 250 #Nm
omega_b = 4800 * (2*np.pi / 60) #rad/s
E_bat = 22.6e3 #Wh
J_mot = 0.025 #kg m^2

''' OTHER '''
g = 9.81 #m/s**2
rho = 1.23 #kg/m**3
S = 2.38 #m**2
C_x = 0.29 #-

**PLOT PARAMETERS**

In [3]:
report_params = {
    # Text Sizes (smaller, readable in print)
    'font.size': 18,            # General default
    'axes.labelsize': 18,       # x and y labels
    'axes.titlesize': 18,       # Title
    'xtick.labelsize': 16,      # Tick numbers
    'ytick.labelsize': 16,
    'legend.fontsize': 16,      # Legend text

    # Line & Marker Geometries (moderate for print)
    'lines.linewidth': 2.0,     # Thicker than default for clarity
    'lines.markersize': 4.0,    # Moderate marker size
    'lines.markeredgewidth': 0,  # Remove marker outline for cleaner look

    # Structural Geometries
    'axes.linewidth': 1.5,       # Thicker spines
    'xtick.major.width': 1.5,    # Thicker ticks
    'ytick.major.width': 1.5,
    'xtick.major.size': 6.0,     # Tick length
    'ytick.major.size': 6.0,

    # Fonts
    'font.family': 'DejaVu Serif',
    'font.serif': ['Times New Roman'],  # Standard report font

    # Figure layout
    'figure.autolayout': True,
    'figure.figsize': (8, 6),     # Smaller for print/report
}
slide_params = {
    # Text Sizes (Large for visibility)
    'font.size': 20,           # General default
    'axes.labelsize': 40,      # x and y labels, can be reduced
    'axes.titlesize': 24,      # Title
    'xtick.labelsize': 20,     # Tick numbers
    'ytick.labelsize': 20,
    'legend.fontsize': 18,     # Legend text

    # axes are dark grey => optional
    'axes.edgecolor': "#7D7D7D",    # Dark grey spines
    'xtick.color': '#7D7D7D',       # Dark grey ticks
    'ytick.color': '#7D7D7D',
    'axes.labelcolor': '#7D7D7D',

    # Line & Marker Geometries (Thick for projectors)
    'lines.linewidth': 5,    # Thicker data lines
    'lines.markersize': 2,    # Much larger markers (default was too small)
    'lines.markeredgewidth': 0,  # Remove marker outline for cleaner look

    # Structural Geometries
    'axes.linewidth': 2.0,     # Thicker spines (box)
    'xtick.major.width': 2.0,  # Thicker ticks
    'ytick.major.width': 2.0,
    'xtick.major.size': 8.0,   # Longer ticks
    'ytick.major.size': 8.0,

    # Slide Aesthetics
    'font.family': 'sans-serif',  # Sans-serif is more legible on slides than serif
    'figure.autolayout': True,   # Similar to tight_layout
    'figure.figsize': (12, 9),   # 4:3 Aspect Ratio by default
}
plt.rcParams.update(report_params)

**ROAD RESISTANCE FORCE AND POWER** (computations)

In [4]:
''' SLOPES '''
theta0 = np.arctan(0) #°
theta5 = np.arctan(0.05) #°
theta10 = np.arctan(0.1) #°

''' SPEED VECTORS '''
v_kmh = np.linspace(0,150,11400) #km/h; 150 is max speed given in litterature
v_ms = v_kmh/3.6

''' GRADING '''
F_grading0 = m*g*np.sin(theta0)
F_grading5 = m*g*np.sin(theta5)
F_grading10 = m*g*np.sin(theta10)

P_grading0 = F_grading0 * v_ms
P_grading5 = F_grading5 * v_ms
P_grading10 = F_grading10 * v_ms

''' AERODYNAMIC DRAG '''
F_AERO = 0.5*rho*S*C_x*(v_ms**2)
P_AERO = F_AERO * v_ms

''' ROLLING RESISTANCE '''
F_RR0  = m*g*(f0 + f2*(v_kmh**2))*np.cos(theta0)
F_RR5  = m*g*(f0 + f2*(v_kmh**2))*np.cos(theta5)
F_RR10 = m*g*(f0 + f2*(v_kmh**2))*np.cos(theta10)

P_RR0  = F_RR0 * v_ms
P_RR5  = F_RR5 * v_ms
P_RR10 = F_RR10 * v_ms

''' TOTAL '''
F_RES0 = F_grading0 + F_AERO + F_RR0
F_RES5 = F_grading5 + F_AERO + F_RR5
F_RES10 = F_grading10 + F_AERO + F_RR10

P_RES0 = P_grading0 + P_AERO + P_RR0
P_RES5 = P_grading5 + P_AERO + P_RR5
P_RES10 = P_grading10 + P_AERO + P_RR10


**ROAD RESISTANCE FORCE AND POWER** (plots)

In [5]:
''' ROLLING RESISTANCE '''
plt.figure()
plt.plot(v_kmh, P_RR0*10**-3, label='Slope = 0%', color = 'blue')
plt.plot(v_kmh, P_RR5*10**-3, label='Slope = 5%', color = 'red')
plt.plot(v_kmh, P_RR10*10**-3, label='Slope = 10%', color = 'green')
plt.xlabel('Speed [km/h]')
plt.ylabel('Rolling resistance power [kW]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/RoadResistance/P_RR.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/RoadResistance/P_RR.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

plt.figure()
plt.plot(v_kmh, F_RR0, label='Slope = 0%', color = 'blue')
plt.plot(v_kmh, F_RR5, label='Slope = 5%', color = 'red')
plt.plot(v_kmh, F_RR10, label='Slope = 10%', color = 'green')
plt.xlabel('Speed [km/h]')
plt.ylabel('Rolling resistance force [N]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/RoadResistance/F_RR.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/RoadResistance/F_RR.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

''' GRADING '''
plt.figure()
plt.plot(v_kmh, P_grading0*10**-3, label='Slope = 0%', color = 'blue')
plt.plot(v_kmh, P_grading5*10**-3, label='Slope = 5%', color = 'red')
plt.plot(v_kmh, P_grading10*10**-3, label='Slope = 10%', color = 'green')
plt.xlabel('Speed [km/h]')
plt.ylabel('Grading resistance power [kW]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/RoadResistance/P_Grading.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/RoadResistance/P_Grading.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

plt.figure()
plt.axhline(y=F_grading0, label='Slope = 0%', color = 'blue')
plt.axhline(y=F_grading5, label='Slope = 5%', color = 'red')
plt.axhline(y=F_grading10, label='Slope = 10%', color = 'green')
plt.xlim(0,150)
plt.xlabel('Speed [km/h]')
plt.ylabel('Grading resistance force [N]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/RoadResistance/F_Grading.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/RoadResistance/F_Grading.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

''' 3 CONTRIBUTIONS '''
plt.figure()
plt.plot(v_kmh, P_AERO*10**-3, label='Aerodynamic drag', color = 'blue')
plt.plot(v_kmh, P_RR0*10**-3, label='Rolling resistance', color = 'red')
plt.plot(v_kmh, P_grading5*10**-3, label='Slope = 5%', color = 'purple')
plt.plot(v_kmh, P_grading10*10**-3, label='Slope = 10%', color = 'green')
plt.xlabel('Speed [km/h]')
plt.ylabel('Power [kW]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/RoadResistance/P_3contrib.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/RoadResistance/P_3contrib.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

plt.figure()
plt.plot(v_kmh, F_AERO, label='Aerodynamic drag', color = 'blue')
plt.plot(v_kmh, F_RR0, label='Rolling resistance', color = 'red')
plt.axhline(y=F_grading5, label='Slope = 5%', color = 'purple')
plt.axhline(y=F_grading10, label='Slope = 10%', color = 'green')
plt.xlim(0,150)
plt.xlabel('Speed [km/h]')
plt.ylabel('Force [N]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/RoadResistance/F_3contrib.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/RoadResistance/F_3contrib.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

''' TOTAL '''
plt.figure()
plt.plot(v_kmh, P_RES0*10**-3, label='Slope = 0%', color = 'blue')
plt.plot(v_kmh, P_RES5*10**-3, label='Slope = 5%', color = 'red')
plt.plot(v_kmh, P_RES10*10**-3, label='Slope = 10%', color = 'green')
plt.xlabel('Speed [km/h]')
plt.ylabel('Total resistive power [kW]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/RoadResistance/P_Tot.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/RoadResistance/P_Tot.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

plt.figure()
plt.plot(v_kmh, F_RES0, label='Slope = 0%', color = 'blue')
plt.plot(v_kmh, F_RES5, label='Slope = 5%', color = 'red')
plt.plot(v_kmh, F_RES10, label='Slope = 10%', color = 'green')
plt.xlabel('Speed [km/h]')
plt.ylabel('Total resistive force [N]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/RoadResistance/F_Tot.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/RoadResistance/F_Tot.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

**MOTOR CURVES** (computations)

In [6]:
omega = np.linspace(1*(2*np.pi)/60, 11400*(2*np.pi)/60, 11400)
rpm = omega*60/(2*np.pi)

P_peak = np.zeros_like(omega)
C_peak = np.zeros_like(omega)
P_nom = np.zeros_like(omega)
C_nom = np.zeros_like(omega)

# Before reaching base rotation speed, torque is constant at maximum value, then
# the power becomes constant after base rotation speed
for i in range(len(omega)):
  if omega[i] <= (P_1p/C_max):
    P_peak[i] = C_max*omega[i]
  else:
    P_peak[i] = P_1p
C_peak = P_peak/omega

# The base speed should be kept beween nominal & peak regimes, so a new torque
# value should be determined by knowing omega_b
C_max_nom = P_1n/omega_b
for j in range(len(omega)):
  if omega[j] <= (P_1n/C_max_nom):
    P_nom[j] = C_max_nom*omega[j]
  else:
    P_nom[j] = P_1n
C_nom = P_nom/omega

# For the next steps, it is more consistent to use nominal regime rather than
# peak one. However, sometimes the 2 of them will be compared

**MOTOR CURVES** (plots)

In [7]:
fig, ax1 = plt.subplots(figsize=(10, 6))

# Left axis : Torques
line1 = ax1.plot(rpm, C_peak, label='Peak torque', color='blue')
line1b = ax1.plot(rpm, C_nom, label='Nominal torque', color='red')
ax1.set_xlabel('Engine rotation speed [RPM]')
ax1.set_ylabel('Torque [Nm]')
ax1.tick_params(axis='y')

# Right axis : Powers
ax2 = ax1.twinx()
line2 = ax2.plot(rpm, P_peak*10**-3, linestyle='--', label='Peak power', color='blue')
line2b = ax2.plot(rpm, P_nom*10**-3, linestyle='--', label='Nominal power', color='red')
ax2.set_ylabel('Power [kW]')
ax2.tick_params(axis='y')
ax2.grid(False) # Either 2 grids mixed, awful

# Scaling
ax1.set_ylim(0, C_max * 1.15)
ax2.set_ylim(0, (P_1p*10**-3) * 1.08)

# Base speed
rpm_base_peak = omega_b * 60 / (2 * np.pi)
line3 = ax1.axvline(x=rpm_base_peak, color='gray', linestyle=':', linewidth=2, label=f'Base speed')

# Legend
lines = line1 + line2 + line1b + line2b + [line3]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='lower center')
#lines, labels, loc='lower center', frameon=True, facecolor='white', framealpha=0.9
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/MotorTraction/MotorCurves.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/MotorTraction/MotorCurves.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

**TRACTIVE FORCE & ACCELERATION CAPABILITY** (computations)

In [8]:
''' TRACTIVE FORCES'''
# Speed vector corresponding to the whole range of motor rotation speed
v_ms_t = (omega*R_e)/i_gear
v_kmh_t = v_ms_t*3.6

# Traction force and power
F_t = (eta_t * C_nom * i_gear)/R_e
P_t = F_t * v_ms

F_t_peak = (eta_t * C_peak * i_gear)/R_e
P_t_peak = F_t_peak * v_ms

''' NET FORCES'''
F_net = F_t - F_RES0
P_net = P_t - P_RES0

F_net5 = F_t - F_RES5
P_net5 = P_t - P_RES5

F_net10 = F_t - F_RES10
P_net10 = P_t - P_RES10

'''ACCELERATION CAPABILITY'''
m_e = m_curb*(1.04 + (J_mot/(m_curb*R_e**2))*i_gear**2) #effective mass

pos = F_net10>0
F_net10_pos = F_net10[pos]

accel_capa = m_e/F_net

accel_capa5 = m_e/F_net5

accel_capa10 = m_e/F_net10_pos
v_acc10 = v_kmh_t[pos]

**TRACTIVE FORCE & ACCELERATION CAPABILITY** (plots)

In [9]:
plt.figure()
plt.plot(v_kmh_t, F_t, label='Nominal', color='blue')
plt.plot(v_kmh_t, F_t_peak, label='Peak', color='red')
plt.xlabel('Speed [km/h]')
plt.ylabel('Tractive force [N]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/MotorTraction/Tractive_Force.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/MotorTraction/Tractive_Force.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

plt.figure()
plt.plot(v_kmh_t, P_t*10**-3, label='Nominal', color='blue')
plt.plot(v_kmh_t, P_t_peak*10**-3, label='Peak', color='red')
plt.xlabel('Speed [km/h]')
plt.ylabel('Tractive power [kW]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/MotorTraction/Tractive_Power.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/MotorTraction/Tractive_Power.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()


#Tractive force
plt.figure()
plt.plot(v_kmh_t, F_t, label='Tractive force', color='blue')
plt.plot(v_kmh, F_RES0, label='Resistive force (0%)', color='red')
plt.plot(v_kmh_t, F_net, label='Net force', color='green')
plt.xlabel('Speed [km/h]')
plt.ylabel('Force [N]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/MotorTraction/TractiveResNet.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/MotorTraction/TractiveResNet.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

plt.figure()
plt.plot(v_kmh_t, F_net5, label='Slope = 5%', color='blue')
plt.plot(v_kmh_t, F_net10, label='Slope = 10%', color='red')
plt.plot(v_kmh_t, F_net, label='Slope = 0%', color='green')
plt.xlabel('Speed [km/h]')
plt.ylabel('Net force [N]')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("figures/MotorTraction/F_Net.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/MotorTraction/F_Net.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

#Acceleration capability
plt.figure()
plt.plot(v_kmh_t, accel_capa, label='Slope = 0%', color='green')
plt.plot(v_kmh_t, accel_capa5, label='Slope = 5%', color='blue')
plt.plot(v_acc10, accel_capa10, label='Slope = 10%', color='red')
plt.xlabel('Speed [km/h]')
plt.ylabel('Acceleration capability [$s^2/m$]')
plt.tight_layout()
plt.grid(True)
plt.legend()
plt.ylim(np.min(accel_capa)-0.5, np.max(accel_capa5)+0.5)
plt.savefig("figures/MotorTraction/Accel_Capa.svg", bbox_inches='tight', dpi=300)
plt.savefig("figures/MotorTraction/Accel_Capa.pdf", bbox_inches='tight', dpi=300)
plt.close()
#plt.show()

**TOP SPEED**

In [10]:
''' MAXIMUM THEORETICAL SPEED'''
max_iter = 100
tol = 1e-4
v_0 = 1 # arbitrary choice
A_v = (m*g*np.sin(theta0)) + (m*g*f0*np.cos(theta0))
B_v = (0.5 * rho * S * C_x) + (m*g*(f2/12.96)*np.cos(theta0))

# Computation of v_max_max and ideal gear ratio
i=0
for i in range(max_iter):
  if i==0:
    v_next = ( ((eta_t * P_1n) - (A_v * v_0)) / B_v )**(1/3)
    v_act = v_next

  else :
    v_next = ( ((eta_t * P_1n) - (A_v * v_act)) / B_v )**(1/3)

    if (np.abs(v_next - v_act) < tol ) :
      break
    else:
      v_act = v_next

v_max_max = v_next
print("Theorical greatest maximum speed: ", v_max_max * 3.6, "km/h")

ideal_i = ((omega_max) * R_e) / v_max_max
print("Ideal gear ratio to reach v_max_max: ", ideal_i)

''' MAXIMUM SPEED WITH IMPOSED GEAR RATIO'''
v_act_bis = v_0
for k in range(max_iter):
    omega_max = 11400*(2*np.pi)/60
    omega_k = min(v_act_bis*i_gear/R_e, omega_max)
    P_k = eta_t * min(C_max_nom*omega_k, P_1n)

    v_new = ((P_k - (A_v * v_act_bis)) / B_v) ** (1/3)

    if abs(v_new - v_act_bis) < tol:
        break
    v_act_bis = v_new
v_power = v_new
v_rpm = ((11400 * (2*np.pi) / 60) * R_e) / i_gear
v_max = min(v_rpm, v_power)

print(f"Real greatest maximum speed: {v_max*3.6:.2f} km/h")

Theorical greatest maximum speed:  193.47294665169997 km/h


NameError: name 'omega_max' is not defined

**MAXIMUM SLOPE**

In [ ]:
mu = np.array([0.2, 0.4, 0.6, 0.7, 0.8, 1]) # choice
print("mu=",mu)


'''MAXIMUM SLOPE IN FIRST GEAR FROM STANSTILL'''
max_slope1 = 100 * ( ((mu * b_cg / L) - f0) / ( 1 - (mu * h_cg / L) ) )
print("Maximum slope in 1st gear : ",max_slope1, "%")

'''MAXIMUM SLOPE AT 80 KM/H '''
F_t_80 = np.interp(80, v_kmh_t, F_t) #need to obtain the value for the exact 80kph speed
F_aero80 = 0.5*rho*S*C_x*(80/3.6)**2
d_80 = (F_t_80 - F_aero80) / (m*g)
max_slope80 = 100 * np.tan( np.arcsin( (d_80 - (f0 * np.sqrt(1 + f0**2 - d_80**2))) / (1 + f0**2) ) )
print("Maximum slope at 80 km/h ",max_slope80, "%")

# VERIFICATION OF TRACTION-LIMITED SLOPE
a = (m*g*(f0+(f2*80**2))) - (mu * m * g * c_cg/L)
b = (m*g) + (mu * m * g * h_cg/L)
max_slope80_adhesion = 100 * np.tan(np.arcsin(-F_aero80/(np.sqrt(a**2 + b**2))) - np.arcsin(a/(np.sqrt(a**2 + b**2))))
print("Verification que la limite de traction est plus grande que la puissance : ", max_slope80_adhesion, ">? 24%")

**ACCELERATION & ELASTICITY **

In [ ]:
''' 50 TO 90 km/h'''
v_50_90 = np.linspace(50/3.6, 90/3.6, 100)
F_net_50_90 = np.interp(v_50_90*3.6, v_kmh_t, F_net)
integral_50_90 = np.trapz(1/F_net_50_90,v_50_90)
time_50_90 = m_e * integral_50_90
print("Acceleration time to go from 50 km/h to 90 km/h :", time_50_90, "s")

'''0 TO 100 km/h'''
v_0_100 = np.linspace(1/3.6, 100/3.6, 100)
F_net_0_100 = np.interp(v_0_100*3.6, v_kmh_t, F_net)
integral_0_100 = np.trapz(1/F_net_0_100,v_0_100)
time_0_100 = m_e * integral_0_100
print("Acceleration time to go from 0 km/h to 100 km/h :", time_0_100, "s")

'''0 TO 400m'''
#aucune idee de v_finale => d'abord calculer temps et distance pour 0-150. Si en dessous de 400m =>
#imposer vitesse constante 150km/h pour derniers metres. Sinon, interpoler avant 150km/h
v_0_150 = np.linspace(1/3.6, 150/3.6, 1000)
F_net_0_150 = np.interp(v_0_150 * 3.6, v_kmh_t, F_net)
time_cumul = m_e * cumulative_trapezoid(1 / F_net_0_150, v_0_150, initial=0)
dist_cumul = m_e * cumulative_trapezoid(v_0_150 / F_net_0_150, v_0_150, initial=0)# pour stocker les valeurs dans un vecteur, utile
# pour interpoler après s'il faut
dist_150 = dist_cumul[-1]
time_150 = time_cumul[-1]

if dist_150 >= 400:
    time_400m = np.interp(400, dist_cumul, time_cumul)
    v_400m_kmh = np.interp(400, dist_cumul, v_0_150 * 3.6)
else:
    dist_restante = 400 - dist_150
    time_400m = time_150 + (dist_restante / (150/3.6))
    v_400m_kmh = 150

print("Acceleration time for 0 to 400m :", time_400m, "s")
print("Speed at 400m :", v_400m_kmh, "km/h")